# Day 10: The Avatar Analogy — Agent Taxonomy in the MFG

*Alpha Flow Research · HongJin HE · July 2026*

---

## Markets as a MMORPG

Imagine financial markets as a massively multiplayer online game (think: World of Warcraft, but everyone's goal is to maximize risk-adjusted return).

Each participant is an **avatar** with:
- A **class** (type): determines their strategy archetype
- **Attributes**: capital, leverage limit, information advantage, time horizon
- **Skills**: the specific alpha strategies they execute
- **Guild membership**: cooperative agreements with other avatars

In the MFG formalism, agent type is encoded in the **individual objective function** $f^i(z, a^i, \mu)$.

## Six Avatar Classes

| Class | Real-World Player | Time Horizon | Alpha Source | Risk Tolerance |
|---|---|---|---|---|
| **Market Maker** | Citadel Securities, Virtu | Seconds | Spread capture, inventory | Low (directional) |
| **Momentum Trader** | CTAs, trend funds | Weeks-months | Price trend | Medium |
| **Value Investor** | Buffett, Ackman | Years | Fundamental mispricing | Low (vol) |
| **Statistical Arb** | Renaissance, Two Sigma | Days | Mean-reversion spreads | Low |
| **Risk Manager** | Pension funds, endowments | Decades | Diversification | Minimal |
| **Central Bank** | Fed, ECB, PBOC | Cycles | Macro stability | None (mandate) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from typing import Callable

@dataclass
class AgentClass:
    name: str
    color: str
    horizon_days: float        # characteristic time horizon
    capital_bn: float          # capital under influence ($B)
    leverage: float            # max leverage
    info_advantage: float      # 0-1 scale
    momentum_sensitivity: float  # positive = momentum, negative = contrarian
    crowding_aversion: float   # Lasry-Lions monotonicity coefficient

agent_classes = [
    AgentClass('Market Maker',    'purple',     0.001, 500,    10.0, 0.9,  0.0, 0.9),
    AgentClass('Statistical Arb', 'steelblue',  5,     200,    3.0,  0.7, -0.5, 0.8),
    AgentClass('Momentum/CTA',    'orange',     60,    300,    2.0,  0.3,  1.0, 0.3),
    AgentClass('Value Investor',  'green',      500,   1000,   1.0,  0.5, -1.0, 0.5),
    AgentClass('Risk Manager',    'teal',       2000,  5000,   0.5,  0.2, -0.2, 0.9),
    AgentClass('Central Bank',    'darkred',    1000,  50000,  1.0,  1.0, -1.5, 1.0),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Horizon vs Capital bubble chart
for a in agent_classes:
    axes[0].scatter(a.horizon_days, a.capital_bn, 
                   s=a.leverage * 80, color=a.color, alpha=0.8,
                   edgecolors='black', linewidth=0.5, zorder=3)
    axes[0].annotate(a.name, (a.horizon_days, a.capital_bn),
                    textcoords='offset points', xytext=(8, 5), fontsize=8)

axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Time horizon (days)')
axes[0].set_ylabel('Capital influenced ($B)')
axes[0].set_title('Agent Taxonomy: Horizon vs Scale\n(bubble size = leverage)')
axes[0].grid(True, alpha=0.3)

# Right: Momentum sensitivity vs Crowding aversion (determines MFG dynamics)
for a in agent_classes:
    axes[1].scatter(a.momentum_sensitivity, a.crowding_aversion,
                   s=200, color=a.color, alpha=0.8, zorder=3,
                   edgecolors='black', linewidth=0.5)
    axes[1].annotate(a.name, (a.momentum_sensitivity, a.crowding_aversion),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)

axes[1].axvline(0, color='gray', ls='--', alpha=0.5)
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.5)
axes[1].set_xlabel('Momentum sensitivity (+ = trend, - = contrarian)')
axes[1].set_ylabel('Crowding aversion (Lasry-Lions monotonicity)')
axes[1].set_title('MFG Parameters by Agent Class\n(High aversion → unique Nash equilibrium)')
axes[1].grid(True, alpha=0.3)

# Add quadrant labels
axes[1].text(-1.4, 0.95, 'Contrarian\n+ Stable', fontsize=8, color='green', style='italic')
axes[1].text(0.6, 0.95, 'Momentum\n+ Stable', fontsize=8, color='orange', style='italic')
axes[1].text(0.6, 0.05, 'Momentum\n+ Unstable (herding)', fontsize=8, color='red', style='italic')

plt.tight_layout()
plt.savefig('../figures/agent_taxonomy.png', dpi=150, bbox_inches='tight')
plt.show()

## Agent Interaction: Who Provides Liquidity to Whom?

The MFG equilibrium emerges from the **interaction** between agent classes. Key interactions:

- **Market Makers → Everyone**: liquidity provision at a spread. They absorb the order flow of other classes.
- **Momentum traders → Value investors**: momentum traders sell to value investors after crashes; buy from them after rallies.
- **Central Banks → Everyone**: policy shocks enter as a common exogenous signal in all agents' information sets.

In the simulation, each agent type generates a **characteristic capital flow pattern** that can be estimated from 13F filings (institutional) and options flow (retail sentiment).

In [ ]:
# Simulate capital flows by agent type over a market cycle
T_cycle = 252 * 2  # 2-year cycle
t_arr = np.arange(T_cycle)

# Simplified market price (bull run + correction)
price = 100 * np.exp(0.12/252 * t_arr + 0.15*np.sin(2*np.pi*t_arr/T_cycle) 
                     + 0.015 * np.cumsum(np.random.randn(T_cycle)))
ret = np.diff(np.log(price))

# Each agent type responds differently to returns
# Momentum: buy after up moves, sell after down
momentum_flow = 500 * np.convolve(ret, np.ones(20)/20, mode='same')
# Value: buy after drops (contrarian)
value_flow = -300 * np.convolve(ret, np.ones(60)/60, mode='same')
# Market maker: always present, flow proportional to vol
mm_flow = 200 * np.abs(np.convolve(ret, np.ones(5)/5, mode='same'))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(t_arr, price, color='black', lw=1.5, label='Market price')
ax1.set_ylabel('Price')
ax1.legend()
ax1.set_title('Capital Flow Patterns by Agent Type Over a Market Cycle')

ax2.plot(t_arr[:-1], momentum_flow, color='orange', lw=1, label='Momentum/CTA flow', alpha=0.8)
ax2.plot(t_arr[:-1], value_flow, color='green', lw=1, label='Value investor flow', alpha=0.8)
ax2.plot(t_arr[:-1], mm_flow, color='purple', lw=1, label='Market maker flow (abs)', alpha=0.8)
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('Capital flow ($M, schematic)')
ax2.set_xlabel('Trading day')
ax2.legend()

plt.tight_layout()
plt.savefig('../figures/agent_capital_flows.png', dpi=150, bbox_inches='tight')
plt.show()

print('The superposition of these flows produces the emergent price dynamics.')
print('This is what the MFG Game Module G computes: the Nash equilibrium of these flows.')